In [ ]:
!pip install -q langchain langchain-community langchain-openai faiss-cpu yfinance gradio sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.1 MB/s eta 0:00:00


In [ ]:
import gradio as gr
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_openai import OpenAI
from langchain.prompts import PromptTemplate
import yfinance as yf
import pandas as pd
from langchain.schema import Document
import os
import getpass
from datetime import datetime
import numpy as np

In [ ]:
evaluation_results = []

In [ ]:
api_key = getpass.getpass("🔑 Enter your OpenAI API key (it will be hidden): ").strip()
if not api_key.startswith("sk-"):
    print("❌ Invalid API key format! It should start with 'sk-'")
    api_key = getpass.getpass("Please re-enter your OpenAI API key: ").strip()
os.environ["OPENAI_API_KEY"] = api_key

🔑 Enter your OpenAI API key (it will be hidden): ··········


In [ ]:
portfolio = pd.DataFrame({
    "Ticker": ["AAPL", "GOOGL", "TSLA", "MSFT", "NVDA"],
    "Buy_Date": ["2024-01-15", "2024-02-01", "2024-01-25", "2024-02-10", "2024-01-05"],
    "Buy_Price": [185.0, 145.0, 220.0, 310.0, 500.0],
    "Quantity": [5, 3, 4, 2, 1]
})

stock_docs = []
for idx, row in portfolio.iterrows():
    symbol = row['Ticker']
    buy_price = row['Buy_Price']
    ticker = yf.Ticker(symbol)
    info = ticker.info
    hist = ticker.history(period="1mo").reset_index()

    price_change = "N/A"
    perf_note = ""
    if len(hist) > 1:
        price_change_val = round((hist.iloc[-1]['Close'] - hist.iloc[0]['Close']) / hist.iloc[0]['Close'] * 100, 2)
        price_change = f"{price_change_val}%"
        if price_change_val < 0:
            perf_note = "The stock has declined this month, which could be dragging your portfolio."
        elif price_change_val < 2:
            perf_note = "The stock has remained mostly flat."
        else:
            perf_note = "The stock is performing well this month."

    summary = f"""
    Company: {symbol}
    Sector: {info.get('sector', 'N/A')}
    Industry: {info.get('industry', 'N/A')}
    Market Cap: {info.get('marketCap', 'N/A')}
    P/E Ratio: {info.get('trailingPE', 'N/A')}
    Dividend Yield: {info.get('dividendYield', 'N/A')}
    Weekly Performance: {price_change}
    Buy Price: {buy_price}
    Recent Close: {hist.iloc[-1]['Close'] if not hist.empty else 'N/A'}
    Performance Note: {perf_note}
    """
    stock_docs.append(Document(page_content=summary, metadata={"ticker": symbol}))

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(stock_docs, embeddings)
retriever = vectorstore.as_retriever()

/tmp/ipython-input-1425593386.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or 

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
template = """
You are StockSensei, an intelligent assistant for financial queries.
Use only the provided context to answer questions. If unsure, say "I couldn't find that in the data."

Context:
{context}

Question: {question}
"""
prompt = PromptTemplate(template=template, input_variables=["context", "question"])

llm = OpenAI(temperature=0, max_tokens=256)

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt}
)


In [ ]:
def evaluate_response(question, generated_answer, ticker_data):
    """
    Evaluates the accuracy of generated answers against actual stock data
    Returns a dictionary of metrics
    """
    metrics = {
        'question': question,
        'answer': generated_answer,
        'timestamp': datetime.now().isoformat(),
        'contains_ticker': False,
        'contains_percentage': False,
        'percentage_accuracy': None,
        'price_accuracy': None,
        'sector_accuracy': None,
        'answer_length': len(generated_answer)
    }

    for ticker in portfolio['Ticker']:
        if ticker in generated_answer:
            metrics['contains_ticker'] = True
            current_ticker = ticker
            break

    if '%' in generated_answer:
        metrics['contains_percentage'] = True
        try:
            # Extract percentage from answer
            percentage_str = generated_answer.split('%')[0].split()[-1]
            reported_pct = float(percentage_str)

            ticker_info = [doc for doc in stock_docs if doc.metadata['ticker'] == current_ticker][0]
            actual_pct = float(ticker_info.page_content.split('Weekly Performance: ')[1].split('%')[0])

            metrics['percentage_accuracy'] = 100 - min(100, abs(reported_pct - actual_pct))
        except:
            pass

    if 'price' in question.lower() or 'compared' in question.lower():
        try:
            price_str = ''.join([c for c in generated_answer.split('$')[1].split()[0] if c.isdigit() or c == '.'])
            reported_price = float(price_str)

            ticker_info = [doc for doc in stock_docs if doc.metadata['ticker'] == current_ticker][0]
            actual_price = float(ticker_info.page_content.split('Recent Close: ')[1].split()[0])

            metrics['price_accuracy'] = 100 - min(100, abs(reported_price - actual_price)/actual_price*100)
        except:
            pass

    return metrics

def calculate_metrics():
    if not evaluation_results:
        return [["No metrics available yet", ""]]

    total = len(evaluation_results)
    correct_ticker = sum(1 for m in evaluation_results if m.get('contains_ticker', False))
    correct_pct = sum(1 for m in evaluation_results if m.get('contains_percentage', False))
    avg_pct_acc = np.mean([m['percentage_accuracy'] for m in evaluation_results if m.get('percentage_accuracy') is not None] or [0])
    avg_price_acc = np.mean([m['price_accuracy'] for m in evaluation_results if m.get('price_accuracy') is not None] or [0])

    return [
        ["Total Questions Processed", total],
        ["Correct Ticker Identification", f"{correct_ticker/total:.1%}"],
        ["Percentage Change Mentioned", f"{correct_pct/total:.1%}"],
        ["Avg Percentage Accuracy", f"{avg_pct_acc:.1f}%"],
        ["Avg Price Accuracy", f"{avg_price_acc:.1f}%"],
        ["Avg Answer Length", f"{np.mean([m['answer_length'] for m in evaluation_results]):.0f} chars"]
    ]


In [ ]:
def stock_sensei(query):
    try:
        response = rag_chain.invoke({"query": query})
        answer = response['result']

        metrics = evaluate_response(query, answer, stock_docs)
        evaluation_results.append(metrics)

        return answer
    except Exception as e:
        error_msg = f"⚠️ Error: {str(e)}. Please check your API key and internet connection."
        evaluation_results.append({
            'question': query,
            'error': str(e),
            'timestamp': datetime.now().isoformat()
        })
        return error_msg

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(primary_hue="green")) as demo:
    gr.Markdown("# 📈 StockSensei - Portfolio Assistant")
    gr.Markdown("### Your Portfolio: AAPL, GOOGL, TSLA, MSFT, NVDA")

    # Display portfolio table
    with gr.Row():
        gr.Dataframe(
            value=portfolio,
            headers=["Ticker", "Buy Date", "Buy Price", "Quantity"],
            row_count=5,
            col_count=(4, "fixed"),
            interactive=False
        )

    with gr.Row():
        input_text = gr.Textbox(
            label="Ask about your portfolio",
            placeholder="e.g., Why is my portfolio down? Which stock is performing best?",
            lines=2
        )

    with gr.Row():
        submit_btn = gr.Button("Analyze", variant="primary")
        clear_btn = gr.Button("Clear")

    output_text = gr.Textbox(label="Analysis", interactive=False, lines=6)

    # Metrics dashboard
    with gr.Accordion("📊 Performance Metrics", open=False):
        gr.Markdown("### System Evaluation Metrics")
        metrics_output = gr.Dataframe(
            headers=["Metric", "Value"],
            value=[["Total Questions Processed", len(evaluation_results)]],
            interactive=False
        )

        update_metrics = gr.Button("Refresh Metrics")
        update_metrics.click(
            fn=calculate_metrics,
            outputs=metrics_output
        )

    gr.Examples(
        examples=[
            "What's the best performing stock in my portfolio?",
            "How is NVDA doing compared to my purchase price?",
            "Which stock has gained the most since I bought it?",
            "How much has my AAPL position gained/lost in percentage terms?",
            "How does MSFT's performance compare to GOOGL's?",
            "Is NVDA performing better than the semiconductor industry average?",
            "Which stock has the highest potential based on recent performance?",
            "Which of my stocks has the lowest P/E ratio?",
            "What's the dividend yield across my entire portfolio?",
            "Which sector in my portfolio is performing best?",
            "How is the tech portion of my portfolio doing?",
            "Are my consumer stocks outperforming my tech stocks?",
            "Which stock in my portfolio is the most volatile?",
        ],
        inputs=input_text
    )

    submit_btn.click(fn=stock_sensei, inputs=input_text, outputs=output_text)
    clear_btn.click(lambda: ["", ""], outputs=[input_text, output_text])

In [ ]:
print("✅ Launching StockSensei interface...")
demo.launch(share=True)

✅ Launching StockSensei interface...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ab516d925d42bba58e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
